# Stage 1 — Faithfulness RD Analysis

This notebook analyses the Risk Difference (RD) results from the faithfulness pipeline.

**Condition contrast**: with-context vs no-context responses on SQuAD questions.

- **Positive RD** → expert more active when the model has context (faithful behaviour)
- **Negative RD** → expert more active when the model lacks context (confabulation behaviour)

Two RD variants are analysed:
- **Frequency-based RD**: based on how often each expert is selected (top-6)
- **Logit-based RD**: based on the mean raw gate logit score per expert

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sys.path.insert(0, os.path.join('..', 'src'))
from rd_utils import load_rd

sns.set_theme(style='whitegrid', font='serif')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

RD_FREQ_PATH   = '/scratch/sc23jc3/results/rd_faithfulness.json'
RD_LOGIT_PATH  = '/scratch/sc23jc3/results/rd_faithfulness_logits.json'
OUT_DIR = 'figures'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
rd_freq  = load_rd(RD_FREQ_PATH)
rd_logit = load_rd(RD_LOGIT_PATH)

layers_sorted = sorted(rd_freq.keys(), key=lambda x: int(x.split('.')[2]))
layer_indices = [int(l.split('.')[2]) for l in layers_sorted]
print(f'Layers: {len(layers_sorted)},  Experts per layer: {len(next(iter(rd_freq.values())))}')

## 1. RD Heatmap — Frequency-based

Each cell shows the RD for one (layer, expert) pair. Red = expert prefers with-context; blue = expert prefers no-context.

In [ ]:
def plot_heatmap(rd_by_layer, layers_sorted, title, filename):
    n_layers  = len(layers_sorted)
    n_experts = len(next(iter(rd_by_layer.values())))
    matrix = np.zeros((n_layers, n_experts))
    for i, layer in enumerate(layers_sorted):
        matrix[i] = rd_by_layer[layer]

    vmax = float(np.max(np.abs(matrix)))
    layer_labels = [int(l.split('.')[2]) for l in layers_sorted]

    fig, ax = plt.subplots(figsize=(18, 7))
    im = ax.imshow(matrix, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('RD', rotation=270, labelpad=15)
    ax.set_xlabel('Expert Index')
    ax.set_ylabel('Layer Index')
    ax.set_title(title)
    ax.set_yticks(range(n_layers))
    ax.set_yticklabels(layer_labels, fontsize=8)
    ax.set_xticks(range(0, n_experts, 8))
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), dpi=300, bbox_inches='tight')
    plt.show()

plot_heatmap(
    rd_freq, layers_sorted,
    'Faithfulness RD Heatmap (Frequency) — With Context vs No Context',
    'faith_rd_heatmap_freq.png'
)

## 2. RD Heatmap — Logit-based

In [ ]:
plot_heatmap(
    rd_logit, layers_sorted,
    'Faithfulness RD Heatmap (Logit) — With Context vs No Context',
    'faith_rd_heatmap_logit.png'
)

## 3. Top Experts by |RD| — Frequency

In [ ]:
def top_experts_table(rd_by_layer, top_n=20):
    rows = []
    for layer, rd in rd_by_layer.items():
        layer_idx = int(layer.split('.')[2])
        for expert_idx, val in enumerate(rd):
            rows.append((abs(val), val, layer_idx, expert_idx))
    rows.sort(reverse=True)
    print(f"{'Rank':>4}  {'Layer':>5}  {'Expert':>6}  {'RD':>10}  Direction")
    print('-' * 45)
    for rank, (abs_val, val, layer, expert) in enumerate(rows[:top_n], 1):
        direction = 'with-context' if val > 0 else 'no-context'
        print(f"{rank:>4}  L{layer:>4}  e{expert:>5}  {val:>+10.6f}  {direction}")

print('=== Top 20 by |RD| — Frequency ===')
top_experts_table(rd_freq)

## 4. Top Experts by |RD| — Logit

In [ ]:
print('=== Top 20 by |RD| — Logit ===')
top_experts_table(rd_logit)

## 5. RD Distribution per Layer

Boxplot showing the spread of RD values across experts for each layer. Layers with high variance have more experts that differentiate between conditions.

In [ ]:
data = [rd_freq[l] for l in layers_sorted]
fig, ax = plt.subplots(figsize=(14, 5))
ax.boxplot(data, positions=layer_indices, widths=0.6,
           patch_artist=True,
           boxprops=dict(facecolor='#d6e4f0', color='#2c3e50'),
           medianprops=dict(color='#e74c3c', linewidth=2),
           whiskerprops=dict(color='#2c3e50'),
           capprops=dict(color='#2c3e50'),
           flierprops=dict(marker='o', markersize=2, alpha=0.4))
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Layer Index')
ax.set_ylabel('RD (Frequency)')
ax.set_title('Distribution of Expert RD Values per Layer — Faithfulness')
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'faith_rd_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

## Discussion

*(Fill in after examining the plots.)*

- Which layers show the strongest RD signal?
- Are high-|RD| experts concentrated in specific layers or spread throughout the network?
- Is the frequency-based RD consistent with the logit-based RD?
- What does this suggest about where faithfulness behaviour is encoded in DeepSeek-V2-Lite?